В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [ ]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 17.3 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [3]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

**Базовые операции с векторами**

In [4]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [5]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

In [8]:
import gensim.downloader as api

info = api.info()

print("Все датасеты:")
for dataset_name in list(info['corpora'].keys())[:10]:
    print(f"- {dataset_name}")

Все датасеты:
- semeval-2016-2017-task3-subtaskBC
- semeval-2016-2017-task3-subtaskA-unannotated
- patent-2017
- quora-duplicate-questions
- wiki-english-20171001
- text8
- fake-news
- 20-newsgroups
- __testing_matrix-synopsis
- __testing_multipart-matrix-synopsis


1. 20-newsgroups: посты из новостных групп по 20 темам, 13 МБ, можно использовать для классификации текстов по признакам, тематического моделирования

2. fake-news: фейк-ньюс и правдивые новости, 30 МБ, бинарная классификация (распознавание фейковых новостей)

3. text8: набор текстов из Википедии, 100 МБ, быстрое и наглядное тестирование различных NLP-алгоритмов, тренировка обучения моделей (как тестовые данные)

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [12]:
dataset = api.load('text8')

[==================================================] 100.0% 31.6/31.6MB downloaded


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [19]:
def get_similar_words(word, topn=10):
  similar_words = w2v_model.most_similar(word, topn=topn)
  print(f"Слова, близкие к '{word}':")

  for i, (similar_word, similarity) in enumerate(similar_words, 1):
    print(f"{i:2d}. {similar_word}, степень близости: {similarity:.3f}")

test_words = ['water', 'person', 'sun', 'food', 'good']
for word in test_words:
  get_similar_words(word)

Слова, близкие к 'water':
 1. natural, степень близости: 0.700
 2. dry, степень близости: 0.677
 3. salt, степень близости: 0.677
 4. clean, степень близости: 0.675
 5. drinking, степень близости: 0.675
 6. gas, степень близости: 0.672
 7. sewage, степень близости: 0.670
 8. supply, степень близости: 0.669
 9. drainage, степень близости: 0.666
10. sea, степень близости: 0.657
Слова, близкие к 'person':
 1. someone, степень близости: 0.794
 2. anyone, степень близости: 0.757
 3. man, степень близости: 0.753
 4. every, степень близости: 0.748
 5. one, степень близости: 0.733
 6. woman, степень близости: 0.723
 7. else, степень близости: 0.716
 8. same, степень близости: 0.713
 9. each, степень близости: 0.704
10. only, степень близости: 0.702
Слова, близкие к 'sun':
 1. moon, степень близости: 0.614
 2. sky, степень близости: 0.613
 3. earth, степень близости: 0.569
 4. light, степень близости: 0.558
 5. microsystems, степень близости: 0.557
 6. bright, степень близости: 0.557
 7. sunshi

3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [20]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [21]:
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    sg=1
)

In [22]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [23]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [24]:
# Найдите слова, похожие на "духовка"
try:
    similar = model.wv.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")

# Найдите слова, похожие на "овощи"
try:
    similar = model.wv.most_similar('овощи', topn=5)
    print("Слова, похожие на 'овощи':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142
Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [25]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [26]:

try:
    similar = ft_model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

try:
    similar = ft_model.wv.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
    print()
except KeyError:
    print("Слово 'духовка' не найдено в словаре")


try:
    similar = ft_model.wv.most_similar('овощи', topn=5)
    print("Слова, похожие на 'овощи':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
    print()
except KeyError:
    print("Слово 'овощи' не найдено в словаре")


Слова, похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622
Слова, похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944

Слова, похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094



7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [27]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('learning')
compare_models('neural')
compare_models('варитб')
compare_models('овосчи')
compare_models('духвка')


Сравнение для слова: 'learning'
  Word2Vec: слово не найдено
  FastText: ['духовка', 'пирог']

Сравнение для слова: 'neural'
  Word2Vec: слово не найдено
  FastText: ['мука', 'травы']

Сравнение для слова: 'варитб'
  Word2Vec: слово не найдено
  FastText: ['варить', 'торт']

Сравнение для слова: 'овосчи'
  Word2Vec: слово не найдено
  FastText: ['начинка', 'овощи']

Сравнение для слова: 'духвка'
  Word2Vec: слово не найдено
  FastText: ['лимон', 'помидоры']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [28]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [29]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [30]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [31]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [32]:
compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [33]:
def highest_similarity(doc1_id):
    highest_match = doc_model.dv.most_similar(f"doc_{doc1_id}", topn=1)[0]
    highest_doc, similarity = highest_match

    top_id = int(highest_doc.split('_')[1])
    print(f"Самый похожий на doc_{doc1_id}: doc_{top_id} (схожесть: {similarity:.4f})")

highest_similarity(1)

Самый похожий на doc_1: doc_0 (схожесть: 0.2735)


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [36]:
vector_sizes = [10, 50, 100]
comparison_words = ['варить', 'духовка', 'овощи']

# Функция для печати похожих слов
def get_similar_words(model_name, model, word, topn=5):
    try:
        similar = model.wv.most_similar(word, topn=topn)
        print(f"  Слова, похожие на '{word}' (модель {model_name}):")
        for w, score in similar:
            print(f"    - {w}: {score:.4f}")
    except KeyError:
        print(f"  Слово '{word}' не найдено в словаре модели {model_name}")

for size in vector_sizes:
    print(f"\nОбучение Word2Vec с vector_size={size}")
    model_current = Word2Vec(
        sentences=cooking_sentences,
        vector_size=size,
        window=3,
        min_count=1,
        workers=2,
        sg=1
    )

    print("Демонстрация похожих слов:")
    for word in comparison_words:
        get_similar_words(f"Word2Vec (размерность {size})", model_current, word)



Обучение Word2Vec с vector_size=10
Демонстрация похожих слов:
  Слова, похожие на 'варить' (модель Word2Vec (размерность 10)):
    - рыба: 0.6654
    - сковорода: 0.6198
    - хлеб: 0.6120
    - готовить: 0.5246
    - яичница: 0.4546
  Слова, похожие на 'духовка' (модель Word2Vec (размерность 10)):
    - взбивать: 0.5916
    - тушить: 0.5880
    - говядина: 0.5443
    - кипятить: 0.5115
    - лимон: 0.5001
  Слова, похожие на 'овощи' (модель Word2Vec (размерность 10)):
    - жарить: 0.7185
    - фольга: 0.7146
    - горшок: 0.6696
    - уголь: 0.6407
    - ингредиенты: 0.5967

Обучение Word2Vec с vector_size=50
Демонстрация похожих слов:
  Слова, похожие на 'варить' (модель Word2Vec (размерность 50)):
    - вино: 0.2398
    - ингредиенты: 0.2172
    - хлеб: 0.1938
    - брокколи: 0.1846
    - кипятить: 0.1711
  Слова, похожие на 'духовка' (модель Word2Vec (размерность 50)):
    - ингредиенты: 0.3199
    - десерт: 0.3064
    - холодильник: 0.2705
    - питание: 0.2243
    - пирог: 0.21

При vector_size=50 модель начинает находить, пожалуй, наиболее осмысленные и релевантные слова-соседей.